### Importing Saved Data

In [1]:
import pandas as pd

df_pivot = pd.read_csv("df_pivot.csv")
df_vanilla_ratios = pd.read_csv("df_vanilla_ratios.csv")
df_baking_powder_ratios = pd.read_csv("df_baking_powder_ratios.csv")

In [2]:
def estimate_eggs(flour, butter):

    fat_ratio = butter / flour

    # --- Base estimate (size-driven) ---
    eggs = int(flour / 100)
    eggs = max(1, min(4, eggs))

    # --- Adjust using fat ratio ---
    if fat_ratio < 0.8:
        eggs += 1
    elif fat_ratio > 1.05:
        eggs -= 1

    # --- Clamp again ---
    eggs = max(1, min(4, eggs))

    return eggs

### Testing the parser

In [3]:
#---Convert vanilla extract values from g/mL to teaspoons and tablespoons for ease of use---

def format_vanilla(vanilla_ml):

    tsp = vanilla_ml / 5
    tsp_rounded = round(tsp)

    tbsp = tsp_rounded // 3
    remaining_tsp = tsp_rounded % 3

    if tbsp > 0 and remaining_tsp > 0:
        return f"{tbsp} tbsp + {remaining_tsp} tsp"
    elif tbsp > 0:
        return f"{tbsp} tbsp"
    else:
        return f"{remaining_tsp} tsp"

### Exploring Spoonacular API as an alternative data source

In [4]:
def format_baking_powder(bp_g):
    tsp = bp_g / 4
    tsp_rounded = round(tsp * 4) / 4  # round to nearest 1/4 tsp

    formats = {
        0.25: "1/4 tsp",
        0.5: "1/2 tsp",
        0.75: "3/4 tsp",
        1.0: "1 tsp"
    }

    if tsp_rounded in formats:
        return formats[tsp_rounded]
    else:
        return f"{tsp_rounded:g} tsp"

### Findings

Spoonacular returned only 1 unique blondie recipe across 5 different search queries, 
which is insufficient for meaningful analysis. The original JSON-LD scraper is retained 
as the primary data collection method.

### Making a dataframe (df) out of the parsed data

In [5]:
def blondie_ingredient_calculator_v3(flour=None, butter=None, sugar=None):

    butter_to_flour = df_pivot["butter_to_flour"].median()
    sugar_to_flour = df_pivot["sugar_to_flour"].median()
    vanilla_to_flour = df_vanilla_ratios["vanilla_to_flour"].median()
    baking_powder_to_flour = df_baking_powder_ratios["bp_to_flour"].median()

    inputs_provided = sum(x is not None and x !=0 for x in [flour, butter, sugar])

    if inputs_provided == 0:
        return "Please provide one ingredient."
    if inputs_provided > 1:
        return "Please provide ONLY one ingredient at a time."

    # --- Convert everything to flour ---
    if butter is not None and butter != 0:
        flour = butter / butter_to_flour
    elif sugar is not None and sugar != 0:
        flour = sugar / sugar_to_flour

    # --- Recompute consistently ---
    butter = flour * butter_to_flour
    sugar = flour * sugar_to_flour

    # --- Vanilla ---
    vanilla_ml = flour * vanilla_to_flour
    vanilla_text = format_vanilla(vanilla_ml)

    # --- Baking Powder ---
    baking_powder_g = flour * baking_powder_to_flour
    baking_powder_text = format_baking_powder(baking_powder_g)

    # --- Eggs ---
    eggs = estimate_eggs(flour, butter)

    result = {
        "flour_g": round(float(flour), 1),
        "baking_powder": baking_powder_text,
        "butter_g": round(float(butter), 1),
        "brown_sugar_g": round(float(sugar), 1),
        "eggs": eggs,
        "vanilla_extract": vanilla_text,
        'optional': "Add 2 tsp cornstarch for extra tenderness, an extra egg yolk for maximum fudginess, or both!"
    }

    ingredient_list = f"""Ingredients:
    - All-purpose Flour: {round(flour, 1)}g
    - Butter (Salted or Unsalted): {round(butter, 1)}g
    - Brown Sugar: {round(sugar, 1)}g
    - Egg(s): {eggs}
    - Vanilla Extract: {vanilla_text}
    - Baking Powder: {baking_powder_text}
    Optional: {result['optional']}
    """
    
    cooking_instructions = """
    Instructions:
    1. Preheat oven to 180°C (350°F). Line a baking pan with parchment paper.
    2. Melt the butter. After it has cooled down, mix with the sugar and vanilla extract until combined.
    3. Beat in the eggs one at a time until fully combined. If using an extra egg yolk, add it in with the eggs.
    4. In a separate container, sift all the dry ingredients together (flour, baking powder, and cornstarch if using).
    5. Fold the dry ingredients into the wet ingredients until just combined.
    6. Fold in chocolate chips, chopped baking chocolate, nuts, or any other add-ins if using. Get creative, folks!
    7. Pour into the pan and bake for 20-25 minutes or until the top is browned and a toothpick inserted one inch from the edge comes out with moist crumbs.
    8. Let the blondie cool at room temperature for 1-2 hours, then refrigerate for at least one hour before serving.
    9. Sprinkle flaky salt on top and enjoy!
    
    Notes:
    - If using unsalted butter, add 1/2 tsp of table salt to the dry ingredients in step 4.
    - Use chopped baking chocolate instead of chocolate chips if you want pools of melted chocolate in the final bake. Mix them both for the best of both worlds!
    - Replacing some brown sugar with white sugar dials back the caramel intensity and gives the top a slight crackle.  
    """

    return result

In [6]:
blondie_ingredient_calculator_v3(butter=150)

{'flour_g': 160.0,
 'baking_powder': '3/4 tsp',
 'butter_g': 150.0,
 'brown_sugar_g': 245.6,
 'eggs': 1,
 'vanilla_extract': '1 tsp',
 'optional': 'Add 2 tsp cornstarch for extra tenderness, an extra egg yolk for maximum fudginess, or both!'}

### Gradio UI and Recipe Generator

Wraps the calculator in a simple interface. Select an ingredient, enter the amount, get a recipe.

In [7]:
import gradio as gr

def blondie_recipe_generator(ingredient, amount):
    if ingredient == "Flour":
        result = blondie_ingredient_calculator_v3(flour=amount)
    elif ingredient == "Butter":
        result = blondie_ingredient_calculator_v3(butter=amount)
    elif ingredient == "Sugar":
        result = blondie_ingredient_calculator_v3(sugar=amount)
    

    ingredient_list = f"""
    Ingredients:
    - All-purpose Flour: {result['flour_g']}g
    - Butter (Salted or Unsalted): {result['butter_g']}g
    - Brown Sugar: {result['brown_sugar_g']}g
    - {'Eggs' if result['eggs'] > 1 else 'Egg'}: {result['eggs']}
    - Vanilla Extract: {result['vanilla_extract']}
    - Baking Powder: {result['baking_powder']}
    Optional: {result['optional']}
    """
    
    cooking_instructions = """
    Instructions:
    1. Preheat oven to 180°C (350°F). Line a baking pan with parchment paper.
    2. Melt the butter. After it has cooled down, mix with the sugar and vanilla extract until combined.
    3. Beat in the eggs one at a time until fully combined. If using an extra egg yolk, add it in with the eggs.
    4. In a separate container, sift all the dry ingredients together (flour, baking powder, and cornstarch if using).
    5. Fold the dry ingredients into the wet ingredients until just combined.
    6. Fold in chocolate chips, chopped baking chocolate, nuts, or any other add-ins if using. Get creative, folks!
    7. Pour into the pan and bake for 20-25 minutes or until the top is browned and a toothpick inserted one inch from the edge comes out with moist crumbs.
    8. Let the blondie cool at room temperature for 1-2 hours, then refrigerate for at least one hour before serving.
    9. Sprinkle flaky salt on top and enjoy!
    
    Notes:
    - If using unsalted butter, add 1/2 tsp of table salt to the dry ingredients in step 4.
    - Use chopped baking chocolate instead of chocolate chips if you want pools of melted chocolate in the final bake. Mix them both for the best of both worlds!
    - Blondies traditionally use all brown sugar, which is responsible for the caramel flavour and browning. If you only have white sugar, that works too — you'll get a delicious blondie with a crinkly top, just without the caramel depth. Replacing some of the brown sugar with white gives you both the caramel flavour and the crinkly top. 
    """
    return ingredient_list + cooking_instructions

interface = gr.Interface(
    fn=blondie_recipe_generator,
    inputs=[
        gr.Dropdown(label="Select Ingredient", choices=["Flour", "Butter", "Sugar"]),
        gr.Number(label="Enter Amount in grams", precision=1, value=None)
    ],
    outputs=gr.Textbox(label="Your Blondie Recipe"),
    title="Blondie Recipe Generator",
    description="Select an ingredient, enter the amount in grams, and get a complete blondie recipe scaled to your kitchen."
)

interface.launch(theme=gr.themes.Ocean())

/Volumes/Samsung_T5/Sweden/PYTHON PADHANAM/PROJECTS/THE PERFECT BLONDIE/the_perfect_blondie/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
